### Fine-tuning LLaMA-3.2-3B on HTP_MD with LoRA for SMILES generation

In [ ]:
!pip install -U transformers

In [ ]:
from huggingface_hub import login, whoami

login(new_session=False)
#print(whoami())

### Adding special tokens and loading the model

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 26.7 MB/s eta 0:00:00


In [ ]:
# Following PolyGen: load the tokenizer and LLaMA model, and add <HIGH> / <LOW> special tokens
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "meta-llama/Llama-3.2-3B-Instruct"

# 4-bit quantization configuration (suitable for Colab T4 / 16GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define label tokens for high / low conductivity
special_tokens = {
    "additional_special_tokens": ["<HIGH>", "<LOW>"]
}

num_added = tokenizer.add_special_tokens(special_tokens)
print("Added special tokens:", num_added)
print("All special tokens:", tokenizer.special_tokens_map)

# If pad_token is not defined, use eos_token as pad
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

# Important: resize embeddings to accommodate the newly added special tokens
model.resize_token_embeddings(len(tokenizer))


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Added special tokens: 2
All special tokens: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>', 'additional_special_tokens': ['<HIGH>', '<LOW>']}


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(128258, 3072)

In [ ]:
# Load the HTP-MD dataset and construct a text column with
# “conditional prefix + p-SMILES”
import pandas as pd

csv_path = "htp_md.csv"

df = pd.read_csv(csv_path, sep="\t")
print(df.head())
print(df["conductivity"].value_counts())

                         mol_smiles  conductivity
0      NC(=O)CSCC(CO[Cu])OC(=O)[Au]             1
1  CCC(F)C(=O)NC(CO[Cu])COC(=O)[Au]             0
2       CCSCCN(CCN[Cu])CCOC(=O)[Au]             1
3     C#CCN(CCOCCO[Cu])CCOC(=O)[Au]             1
4     CCC(COC(=O)[Au])C(=O)NCCO[Cu]             0
conductivity
1    5704
0    5704
Name: count, dtype: int64


In [ ]:
# Following PolyGen dataset.py: construct text by repeating
# <HIGH> or <LOW> five times, then appending the p-SMILES
PREFIX_LEN = 5   # Align with length=5 used in PolyGen

def row_to_text(row):
    label = row["conductivity"]
    if label == 1:
        tok = "<HIGH>"
    else:
        tok = "<LOW>"

    prefix = " ".join([tok] * PREFIX_LEN)     # "<HIGH> <HIGH> <HIGH> <HIGH> <HIGH>"
    # Simple concatenation: prefix + space + p-SMILES
    return prefix + " " + str(row["mol_smiles"])

df["text"] = df.apply(row_to_text, axis=1)
df[["mol_smiles", "conductivity", "text"]].head()

,mol_smiles,conductivity,text
0,NC(=O)CSCC(CO[Cu])OC(=O)[Au],1,<HIGH> <HIGH> <HIGH> <HIGH> <HIGH> NC(=O)CSCC(...
1,CCC(F)C(=O)NC(CO[Cu])COC(=O)[Au],0,<LOW> <LOW> <LOW> <LOW> <LOW> CCC(F)C(=O)NC(CO...
2,CCSCCN(CCN[Cu])CCOC(=O)[Au],1,<HIGH> <HIGH> <HIGH> <HIGH> <HIGH> CCSCCN(CCN[...
3,C#CCN(CCOCCO[Cu])CCOC(=O)[Au],1,<HIGH> <HIGH> <HIGH> <HIGH> <HIGH> C#CCN(CCOCC...
4,CCC(COC(=O)[Au])C(=O)NCCO[Cu],0,<LOW> <LOW> <LOW> <LOW> <LOW> CCC(COC(=O)[Au])...


In [ ]:
# split train / validation, and convert to HuggingFace Dataset
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df[["text"]],
    test_size=0.1,
    random_state=42,
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))

raw_datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
})

raw_datasets


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 10267
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1141
    })
})

### Tokenizer

In [ ]:
# Tokenize the data and prepare input_ids / labels for language model training.
# Use a fixed maximum length, e.g., 80 tokens.
MAX_LENGTH = 80

def tokenize_fn(examples):
    # Only process the "text" column
    outputs = tokenizer(
        examples["text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )
    # For an autoregressive LM, labels are the same as input_ids
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

tokenized_datasets = raw_datasets.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
)

tokenized_datasets


Map:   0%|          | 0/10267 [00:00<?, ? examples/s]

Map:   0%|          | 0/1141 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10267
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1141
    })
})

### Add LoRA adapter

In [ ]:
# Wrap the model with LoRA to reduce the number of trainable parameters
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    # Common target modules for the LLaMA family
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,069,824 || trainable%: 0.7511


In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
output_dir = "/content/drive/MyDrive/llama-polygen-htpmd-lora"
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


### Train model

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint

# Find the last checkpoint (it will automatically pick checkpoint-963)
last_ckpt = get_last_checkpoint(output_dir)
print("Resume from:", last_ckpt)

# Set the number of epochs to 5 (train for a total of 5 epochs)
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    bf16=True,
    tf32=True,
    num_train_epochs=5,
    learning_rate=2e-4,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
)

model.enable_input_require_grads()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
)

# Resume from the previous checkpoint and continue until a total of 5 epochs
trainer.train(resume_from_checkpoint=last_ckpt)

Resume from: /content/drive/MyDrive/llama-polygen-htpmd-lora/checkpoint-963


/tmp/ipython-input-778507346.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss
1000,1.127000,1.138222
1100,1.128100,1.138405
1200,1.129900,1.138776
1300,1.122600,1.137689
1400,1.120100,1.138244
1500,1.121800,1.137964
1600,1.123300,1.137056


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetunin

TrainOutput(global_step=1605, training_loss=0.45001511677774686, metrics={'train_runtime': 1708.4921, 'train_samples_per_second': 30.047, 'train_steps_per_second': 0.939, 'total_flos': 7.00553035997184e+16, 'train_loss': 0.45001511677774686, 'epoch': 5.0})

In [ ]:
trainer.save_model(output_dir)        # contains LoRA weights
tokenizer.save_pretrained(output_dir)

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


('/content/drive/MyDrive/llama-polygen-htpmd-lora/tokenizer_config.json',
 '/content/drive/MyDrive/llama-polygen-htpmd-lora/special_tokens_map.json',
 '/content/drive/MyDrive/llama-polygen-htpmd-lora/chat_template.jinja',
 '/content/drive/MyDrive/llama-polygen-htpmd-lora/tokenizer.json')

### Load model, tokenizer and LoRA adapter

In [ ]:
# Reload the fine-tuned model and tokenizer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

peft_model_dir = "/content/drive/MyDrive/llama-polygen-htpmd-lora"  # your saved directory

# 1. Load the tokenizer (it already includes the <HIGH>, <LOW> special tokens you added)
tokenizer = AutoTokenizer.from_pretrained(peft_model_dir)

# 2. Load the base LLaMA model
base_model_name = "meta-llama/Llama-3.2-3B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
base_model.resize_token_embeddings(len(tokenizer))

# 3. Load the LoRA adapter back onto the base model
model = PeftModel.from_pretrained(base_model, peft_model_dir)
model.eval()

# 4. Fallback for pad_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", len(tokenizer))
print("Embedding size:", base_model.get_input_embeddings().weight.shape[0])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Vocab size: 128258
Embedding size: 128258


### Generate SMILES

In [ ]:
PREFIX_LEN = 5  # Keep consistent with the value used when building the dataset

def sample_llama_smiles(cond="high", n=100, max_new_tokens=60):
    """
    cond: "high" or "low"
    Returns n p-SMILES generated by the fine-tuned LLaMA model (without the prefix tokens)
    """
    cond_token = "<HIGH>" if cond.lower().startswith("h") else "<LOW>"
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    samples = []

    while len(samples) < n:
        # Construct the prefix: <HIGH> <HIGH> ... or <LOW> <LOW> ...
        prefix = " ".join([cond_token] * PREFIX_LEN)

        inputs = tokenizer(prefix, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=1.0,
                top_p=0.95,
                pad_token_id=pad_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode the full output and remove the prefix, keeping only the p-SMILES part
        full_text = tokenizer.decode(out_ids[0], skip_special_tokens=True)

        # Remove the prefix (strip first to avoid extra spaces)
        if full_text.startswith(prefix):
            tail = full_text[len(prefix):].strip()
        else:
            # Occasionally the tokenizer inserts extra spaces; split by the prefix as a fallback
            tail = full_text.split(cond_token * PREFIX_LEN)[-1].strip()

        # Keep only the first p-SMILES segment: cut at whitespace or newline
        first_piece = tail.split()[0]
        samples.append(first_piece)

    return samples[:n]

In [ ]:
# Generate 100 “high-conductivity” p-SMILES
llama_high_psmiles = sample_llama_smiles(cond="high", n=100)
llama_high_psmiles[:10]

# Generate 100 “low-conductivity” p-SMILES
llama_low_psmiles = sample_llama_smiles(cond="low", n=100)
llama_low_psmiles[:10]

['CC(N[Cu])C(COC(=O)[Au])NC(=O)C(C)(C)C',
 'CSCCN(CCO[Cu])CCOC(=O)[Au]',
 'O=C([Au])OCCOCCN[Cu]',
 'COCCCCNCC(CO[Cu])OC(=O)[Au]',
 'CN(CCCCOC(=O)[Au])CCOCCO[Cu]',
 'O=C(CCO[Cu])NCCNCCOC(=O)[Au]',
 'COCCCNC(=O)C(CN[Cu])NC(=O)[Au]',
 'CC(O[Cu])C(CNC(=O)CC(F)F)OC(=O)[Au]',
 'O=C([Au])NCCOCC(=O)NCCN[Cu]',
 'CC(CN(CCC#N)CCO[Cu])OC(=O)[Au]']

In [ ]:
import json

with open("../../data/generated/llama_tuned/llama_high_tuned.json", "w") as f:
    json.dump(llama_high_psmiles, f)

with open("../../data/generated/llama_tuned/llama_low_tuned.json", "w") as f:
    json.dump(llama_low_psmiles, f)
